# Phase 5 — NB3: Joint Evaluation (SemEval 2014)

**Goal:** End-to-end evaluation: Stage 1 predict categories → Stage 2 predict sentiment.
Compare retrieval vs no-retrieval.

**Input:**
- `lcminhc/p5-nb1-stage1` — Stage 1 ckpts + processed data
- `lcminhc/p5-nb2-stage2` — Stage 2 ckpts
- `lcminhc/p5-embed-v4` — embedding ckpt (for retrieval)

**Output:** Metrics report (Category F1, Joint F1, Sentiment Acc|Correct Cat)

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
gc.collect()
torch.cuda.empty_cache()

## 0b. Configuration

Choose Stage 1 checkpoint based on NB1 results.

In [ ]:
# ===== CONFIGURATION =====
# Set based on NB1 results: 'asl' or 'cataware'
STAGE1_VARIANT = 'asl'

if STAGE1_VARIANT == 'asl':
    STAGE1_CONFIG = 'configs/stage1_2014.yaml'
    STAGE1_CKPT_NAME = 'stage1_2014_best.pt'
else:
    STAGE1_CONFIG = 'configs/stage1_2014_cataware.yaml'
    STAGE1_CKPT_NAME = 'stage1_2014_cataware_best.pt'

print(f'Stage 1: {STAGE1_VARIANT} -> {STAGE1_CONFIG}, {STAGE1_CKPT_NAME}')

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}', f'/kaggle/input/datasets/lcminhc/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1')
NB2 = find_input('p5-nb2-stage2')
EMB = find_input('p5-embed-v4')

print(f'NB1: {NB1} -> {os.listdir(NB1)}')
print(f'NB2: {NB2} -> {os.listdir(NB2)}')
print(f'EMB: {EMB} -> {os.listdir(EMB)[:5]}...')

# Stage 1 checkpoint
os.makedirs('checkpoints/stage1', exist_ok=True)
shutil.copy(f'{NB1}/{STAGE1_CKPT_NAME}', 'checkpoints/stage1/best.pt')
print(f'Stage 1 ckpt: {STAGE1_CKPT_NAME}')

# Stage 2 checkpoints
os.makedirs('checkpoints/stage2_2014', exist_ok=True)
os.makedirs('checkpoints/stage2_2014_noret', exist_ok=True)
shutil.copy(f'{NB2}/stage2_2014_best.pt', 'checkpoints/stage2_2014/best.pt')
shutil.copy(f'{NB2}/stage2_2014_noret_best.pt', 'checkpoints/stage2_2014_noret/best.pt')

# Embedding checkpoint
os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_best.pt', 'checkpoints/embedding_2014/best.pt')

# Processed data
os.makedirs('data/processed', exist_ok=True)
shutil.copy(f'{NB1}/category_detection.jsonl', 'data/processed/category_detection.jsonl')
shutil.copy(f'{NB1}/sentiment_records.jsonl', 'data/processed/sentiment_records.jsonl')

# Build FAISS index (must match NB2 training index)
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

print('\nAll files wired:')
for p in ['checkpoints/stage1/best.pt',
          'checkpoints/stage2_2014/best.pt',
          'checkpoints/stage2_2014_noret/best.pt',
          'checkpoints/embedding_2014/best.pt',
          'indexes/train.faiss',
          'data/processed/category_detection.jsonl',
          'data/processed/sentiment_records.jsonl']:
    size = os.path.getsize(p) / 1e6
    print(f'  {p}: {size:.1f} MB')

## 1. Evaluate WITH Retrieval

In [ ]:
!python scripts/05_evaluate_joint.py \
    --stage1_ckpt checkpoints/stage1/best.pt \
    --stage2_ckpt checkpoints/stage2_2014/best.pt \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --index_dir indexes/ \
    --stage1_config {STAGE1_CONFIG} \
    --stage2_config configs/stage2_2014.yaml \
    --retrieval_config configs/retrieval_v2.yaml \
    --pred_strategy all

In [ ]:
if os.path.exists('logs/joint_eval_retrieval.md'):
    with open('logs/joint_eval_retrieval.md') as f:
        print(f.read())
elif os.path.exists('logs/joint_eval_results.md'):
    shutil.copy('logs/joint_eval_results.md', 'logs/joint_eval_retrieval.md')
    with open('logs/joint_eval_retrieval.md') as f:
        print(f.read())

## 2. Evaluate WITHOUT Retrieval

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/05_evaluate_joint.py \
    --stage1_ckpt checkpoints/stage1/best.pt \
    --stage2_ckpt checkpoints/stage2_2014_noret/best.pt \
    --stage1_config {STAGE1_CONFIG} \
    --stage2_config configs/stage2_2014_noret.yaml \
    --no_retrieval \
    --pred_strategy all

In [ ]:
if os.path.exists('logs/joint_eval_noret.md'):
    with open('logs/joint_eval_noret.md') as f:
        print(f.read())
elif os.path.exists('logs/joint_eval_results.md'):
    shutil.copy('logs/joint_eval_results.md', 'logs/joint_eval_noret.md')
    with open('logs/joint_eval_noret.md') as f:
        print(f.read())

## 3. Results Comparison

In [ ]:
print('=' * 60)
print(f'RETRIEVAL (Phase 2a) — Stage 1: {STAGE1_VARIANT}')
print('=' * 60)
if os.path.exists('logs/joint_eval_retrieval.md'):
    with open('logs/joint_eval_retrieval.md') as f:
        print(f.read())

print('\n' + '=' * 60)
print(f'NO-RETRIEVAL — Stage 1: {STAGE1_VARIANT}')
print('=' * 60)
if os.path.exists('logs/joint_eval_noret.md'):
    with open('logs/joint_eval_noret.md') as f:
        print(f.read())

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb3'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')